In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q sentencepiece datasets
import torch
print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/otto_gpt'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/ckpt', exist_ok=True)
print("프로젝트 폴더:", PROJECT_DIR)
print("내용:", os.listdir(PROJECT_DIR))

In [ ]:
from datasets import load_dataset

RAW_TXT = f'{PROJECT_DIR}/data/corpus_wiki.txt'

if not os.path.exists(RAW_TXT):
    print("한국어 위키 다운로드 중... (처음엔 50,000 문서로 시작)")
    ds = load_dataset("wikimedia/wikipedia", "20231101.ko",
                      split="train", streaming=True)
    n_docs = 50_000   
    with open(RAW_TXT, 'w', encoding='utf-8') as f:
        for i, doc in enumerate(ds):
            if i >= n_docs:
                break
            text = doc['text'].strip()
            if len(text) > 100:        
                f.write(text + '\n\n')
            if (i+1) % 10000 == 0:
                print(f"  {i+1} 문서 저장")
    print("완료:", RAW_TXT)
else:
    print("이미 존재:", RAW_TXT)

size_mb = os.path.getsize(RAW_TXT) / 1e6
print(f"코퍼스 크기: {size_mb:.1f} MB")

In [ ]:
import glob

all_txt_files = glob.glob(f'{PROJECT_DIR}/data/*.txt')
print("발견된 데이터 파일:")
for p in all_txt_files:
    print(f"  {os.path.basename(p)}: {os.path.getsize(p)/1e6:.1f} MB")

MERGED_TXT = f'{PROJECT_DIR}/data/_merged.txt'
with open(MERGED_TXT, 'w', encoding='utf-8') as out:
    for p in all_txt_files:
        if os.path.basename(p) == '_merged.txt':
            continue
        with open(p, encoding='utf-8') as f:
            out.write(f.read() + '\n')
print(f"\n합본: {MERGED_TXT} ({os.path.getsize(MERGED_TXT)/1e6:.1f} MB)")

In [ ]:
import sentencepiece as spm

SPM_PREFIX = f'{PROJECT_DIR}/tokenizer_ko'
VOCAB_SIZE = 16000

if not os.path.exists(SPM_PREFIX + '.model'):
    print("토크나이저 학습 중... (몇 분 소요)")
    spm.SentencePieceTrainer.train(
        input=MERGED_TXT,
        model_prefix=SPM_PREFIX,
        vocab_size=VOCAB_SIZE,
        model_type='bpe',
        character_coverage=0.9995,
        
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        user_defined_symbols=['<sep>', '<pad2>'],
        input_sentence_size=1_000_000,   
        shuffle_input_sentence=True,
    )
    print("완료:", SPM_PREFIX + '.model')
else:
    print("이미 존재:", SPM_PREFIX + '.model')

sp = spm.SentencePieceProcessor(model_file=SPM_PREFIX + '.model')
print("\n어휘 크기:", sp.get_piece_size())
sample = "인공지능은 사람처럼 학습하는 기술이다."
ids = sp.encode(sample)
print(f"\n원문: {sample}")
print(f"토큰: {[sp.id_to_piece(i) for i in ids]}")
print(f"ID:   {ids}")
print(f"복원: {sp.decode(ids)}")

In [ ]:
import numpy as np

TRAIN_BIN = f'{PROJECT_DIR}/data/train.bin'
VAL_BIN = f'{PROJECT_DIR}/data/val.bin'

if not (os.path.exists(TRAIN_BIN) and os.path.exists(VAL_BIN)):
    print("코퍼스 토크나이징 중...")
    
    all_ids = []
    with open(MERGED_TXT, encoding='utf-8') as f:
        buf = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            ids = sp.encode(line)
            buf.extend(ids)
            buf.append(sp.eos_id())
            if len(buf) > 1_000_000:
                all_ids.append(np.array(buf, dtype=np.uint16))
                buf = []
        if buf:
            all_ids.append(np.array(buf, dtype=np.uint16))
    data = np.concatenate(all_ids)
    print(f"총 토큰 수: {len(data):,}")

    n = len(data)
    split = int(n * 0.9)
    data[:split].tofile(TRAIN_BIN)
    data[split:].tofile(VAL_BIN)
    print(f"train: {split:,} 토큰 → {TRAIN_BIN}")
    print(f"val:   {n-split:,} 토큰 → {VAL_BIN}")
else:
    print("이미 존재. 토큰 수 확인:")
    print(f"  train: {os.path.getsize(TRAIN_BIN)//2:,}")
    print(f"  val:   {os.path.getsize(VAL_BIN)//2:,}")

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = 16000
    block_size: int = 512     
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 624
    dropout: float = 0.1
    bias: bool = False

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.c_attn = nn.Linear(cfg.n_embd, 3*cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        self.n_head, self.n_embd, self.dropout = cfg.n_head, cfg.n_embd, cfg.dropout

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4*cfg.n_embd, bias=cfg.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4*cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp = MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(cfg.vocab_size, cfg.n_embd),   
            wpe=nn.Embedding(cfg.block_size, cfg.n_embd),   
            drop=nn.Dropout(cfg.dropout),
            h=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f=nn.LayerNorm(cfg.n_embd, bias=cfg.bias),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight   
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2*cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        n = sum(p.numel() for p in self.parameters())
        return n - self.transformer.wpe.weight.numel()

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-1)
            return logits, loss
        logits = self.lm_head(x[:, [-1], :])
        return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("GPT 모델 정의 완료")

In [ ]:
config = GPTConfig(
    vocab_size=VOCAB_SIZE,
    block_size=512,
    n_layer=10,
    n_head=12,
    n_embd=624,
    dropout=0.1,
)

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 38:        
        batch_size = 24
        grad_accum = 2
    elif vram_gb >= 22:      
        batch_size = 12
        grad_accum = 4
    else:                    
        batch_size = 6
        grad_accum = 8
else:
    batch_size, grad_accum = 2, 1

learning_rate = 3e-4
min_lr = 3e-5
warmup_iters = 200
max_iters = 5000          
eval_interval = 250       
eval_iters = 50
weight_decay = 0.1
grad_clip = 1.0
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'

print(f"디바이스: {device}, dtype: {dtype}")
print(f"batch_size={batch_size}, grad_accum={grad_accum}")
print(f"유효 배치: {batch_size*grad_accum*config.block_size:,} 토큰/step")

m_tmp = GPT(config)
print(f"모델 파라미터: {m_tmp.num_params()/1e6:.1f}M")
del m_tmp

In [ ]:
train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode='r')
val_data = np.memmap(VAL_BIN, dtype=np.uint16, mode='r')

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - config.block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+config.block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+config.block_size].astype(np.int64)) for i in ix])
    if device == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print("배치 shape:", xb.shape, yb.shape)
print("샘플 디코드:", sp.decode(xb[0][:30].tolist()))

In [ ]:
import time

CKPT_PATH = f'{PROJECT_DIR}/ckpt/otto_gpt.pt'

model = GPT(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                              betas=(0.9, 0.95), weight_decay=weight_decay)
scaler = torch.amp.GradScaler(enabled=(dtype == 'float16'))

iter_num = 0
best_val_loss = float('inf')
if os.path.exists(CKPT_PATH):
    print(f"✓ 체크포인트 발견 → 이어서 학습: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=device)
    
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    iter_num = ckpt['iter_num']
    best_val_loss = ckpt['best_val_loss']
    print(f"  복원: iter_num={iter_num:,}, best_val_loss={best_val_loss:.4f}")
else:
    print("새로 학습 시작 (체크포인트 없음)")

print(f"학습 파라미터: {model.num_params()/1e6:.1f}M\n")

def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    if it > max_iters:
        return min_lr
    ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with torch.amp.autocast(device_type='cuda', dtype=getattr(torch, dtype)) if device=='cuda' else torch.no_grad():
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def save_checkpoint(val_loss):
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'config': config.__dict__,
        'iter_num': iter_num,
        'best_val_loss': val_loss,
    }, CKPT_PATH)
    print(f"    💾 체크포인트 저장 (val_loss={val_loss:.4f})")

print("학습 시작...\n")
model.train()
t0 = time.time()
target_iter = iter_num + max_iters    

while iter_num < target_iter:
    lr = get_lr(iter_num)
    for g in optimizer.param_groups:
        g['lr'] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    for micro in range(grad_accum):
        X, Y = get_batch('train')
        if device == 'cuda':
            with torch.amp.autocast(device_type='cuda', dtype=getattr(torch, dtype)):
                _, loss = model(X, Y)
                loss = loss / grad_accum
        else:
            _, loss = model(X, Y)
            loss = loss / grad_accum
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()

    if iter_num % 50 == 0:
        dt = time.time() - t0
        print(f"iter {iter_num:5d} | loss {accum_loss:.4f} | lr {lr:.2e} | {dt:.1f}s")
        t0 = time.time()

    if iter_num % eval_interval == 0 and iter_num > 0:
        losses = estimate_loss()
        print(f"  >> eval | train {losses['train']:.4f} | val {losses['val']:.4f}")
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            save_checkpoint(best_val_loss)

    iter_num += 1

final_losses = estimate_loss()
print(f"\n학습 종료. 최종 val_loss: {final_losses['val']:.4f}")
if final_losses['val'] < best_val_loss:
    best_val_loss = final_losses['val']
save_checkpoint(best_val_loss)
print(f"누적 학습 step: {iter_num:,}")

In [ ]:
def generate_text(prompt, max_new_tokens=100, temperature=0.8, top_k=50):
    model.eval()
    ids = sp.encode(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        if device == 'cuda':
            with torch.amp.autocast(device_type='cuda', dtype=getattr(torch, dtype)):
                out = model.generate(x, max_new_tokens, temperature, top_k)
        else:
            out = model.generate(x, max_new_tokens, temperature, top_k)
    return sp.decode(out[0].tolist())

for prompt in ["인공지능은", "한국의 역사는", "오늘 날씨가"]:
    print(f"[입력] {prompt}")
    print(f"[생성] {generate_text(prompt, max_new_tokens=80)}\n")

In [ ]:
print("="*50)
print("otto-GPT 현재 상태")
print("="*50)
print(f"파라미터:      {model.num_params()/1e6:.1f}M")
print(f"누적 학습 step: {iter_num:,}")
print(f"최고 val_loss:  {best_val_loss:.4f}")
print(f"어휘 크기:      {sp.get_piece_size():,}")
print(f"체크포인트:     {CKPT_PATH}")
print(f"토크나이저:     {SPM_PREFIX}.model")
print("="*50)
print("\n다음에 이어서 학습하려면: 이 노트북을 다시 열고 셀을 순서대로 실행하세요.")